In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DateType
from datetime import datetime
import re

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("scottish_housing_ws.bronze.housebuilding")

df_bronze.select("featuretype").distinct().show(10, truncate=False)
df_bronze.select("housing_sector").distinct().show(10, truncate=False)
df_bronze.select("build_type").distinct().show(10, truncate=False)

In [0]:
from pyspark.sql.types import DateType
from pyspark.sql.functions import udf

def parse_quarter_date(date_code):
    match = re.match(r"(\d{4})-Q(\d)", date_code)
    if not match:
        return None
    year = int(match.group(1))
    quarter = int(match.group(2))
    month = {1: 1, 2: 4, 3: 7, 4: 10}[quarter]
    return datetime(year, month, 1).date()

parse_quarter_udf = udf(parse_quarter_date, DateType())

In [0]:
df_silver = (
    df_bronze
    .filter(col("featuretype").isin("Country", "Council Area"))
    .withColumn("report_date", parse_quarter_udf(col("datecode")))
    .withColumnRenamed("datecode", "year_quarter")
    .withColumnRenamed("featurecode", "area_code")
    .withColumnRenamed("featurename", "area_name")
    .withColumnRenamed("featuretype", "area_type")
    .withColumnRenamed("value", "dwellings")
    .drop("measurement", "units")
    .select(
        "report_date", "year_quarter", "area_code", "area_name", "area_type",
        "housing_sector", "build_type", "dwellings"
    )
)

df_silver.printSchema()

In [0]:
df_silver.orderBy("report_date", "area_type", "housing_sector", "build_type").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import min, max, count
df_silver.agg(
    count("*").alias("row_count"),
    min("report_date").alias("earliest"),
    max("report_date").alias("latest")
).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.housebuilding")
)

In [0]:
# Filtered to Country and Council Area geographies (Health Board Area dropped -- no join key).
# All four housing sectors retained: All, Private, Local Authority, Housing Association.
# report_date is first day of each quarter.
# Fully qualified table name on write due to UDF catalog reset gotcha.